In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
logger = get_logger()

In [0]:

dim_products_df  = spark.read.table("az_ci_de_dbs.ecom_dev.dim_products")
dim_customers_df = spark.read.table("az_ci_de_dbs.ecom_dev.dim_customers")
fact_orders_df   = spark.read.table("az_ci_de_dbs.ecom_dev.fact_orders")


In [0]:
def build_enriched_orders(fact_df, dim_customers_df, dim_products_df):
    """Join fact orders with dimension tables and return enriched DataFrame."""
    enriched = (
        fact_df.alias("o")
        .join(F.broadcast(dim_customers_df).alias("c"), on="customer_id", how="inner")
        .join(F.broadcast(dim_products_df).alias("p"),  on="product_id",  how="inner")
        .select(
            "o.*",
            "c.customer_name",
            "c.country",
            "p.category",
            "p.sub_category",
        )
    )
    return enriched.dropDuplicates()


def build_profit_by_year(enriched_df):
    """Aggregate profit by year, category, sub_category, customer."""
    profit = (
        enriched_df
        .withColumn("year", F.year(F.col("order_date")))
        .groupBy("year", "category", "sub_category", "customer_name")
        .agg(F.round(F.sum("profit"), 2).alias("total_profit"))
    )
    return profit.dropDuplicates()


def main_enrich():
    try:
        enriched_orders = build_enriched_orders(fact_orders_df, dim_customers_df, dim_products_df)
        delta_writer(enriched_orders, "az_ci_de_dbs.ecom_dev.enriched_orders")

        profit_by_year = build_profit_by_year(
            spark.read.table("az_ci_de_dbs.ecom_dev.enriched_orders")
        )
        delta_writer(profit_by_year, "az_ci_de_dbs.ecom_dev.profit_by_year")

        logger.info("Processing completed successfully.")

    except Exception as e:
        logger.error(f"Error during processing: {e}")
